# Run an election from a hand-built `data` dictionary

## 1. The `data` dictionary

Every response function and voting rule consumes the same dict:

```python
data = {
    "beliefs":     {"mean": ..., "precision": ...},   # (n_agents, n_pref)
    "preferences": {"mean": ..., "precision": ...},   # (n_agents, n_pref)
    "candidates":  {"mean": ..., "precision": ...},   # (n_candidates, n_pref)
}
```

In [ ]:
from typing import Any

import jax
import jax.numpy as jnp
import tqdm
from pyparsing import Dict

from eci.decision import response_function_pref
from eci.voting import _vote_plurality, _vote_quadratic

key = jax.random.PRNGKey(0)

In [ ]:
n_agents, n_candidates, n_pref = 200, 4, 3

# --- preferences: ideal points ~ N(0, 1.5), held with precision 1.0 ---
key, k = jax.random.split(key)
pref_mean = jax.random.normal(k, (n_agents, n_pref)) * 1.5
pref_precision = jnp.full((n_agents, n_pref), 1.0)

# --- candidates: fixed positions, extreme (C0, C3) to central (C1, C2) ---
cand_mean = jnp.array([[-3.0], [-1.0], [1.0], [3.0]])  # (n_candidates, n_pref)
cand_precision = jnp.full((n_candidates, n_pref), 5.0)


data = {
    "preferences": {"mean": pref_mean, "precision": pref_precision},
    "candidates": {"mean": cand_mean, "precision": cand_precision},
}

data

In [ ]:
def run_n_simulation(
    func,
    data,
    response_function,
    key,
    n_simulations: int,
    *args,
    **kwargs,
):
    """Run ``n_simulations`` simulations sequentially, returning a dict.

    TODO: replace the Python loop with ``jax.vmap`` over PRNG keys.
    """
    all_results: Dict[int, Any] = {}
    current_key = key
    for i in tqdm.tqdm(range(n_simulations), desc="Running Simulations"):
        current_key, subkey = jax.random.split(current_key)
        all_results[i] = func(data, response_function, subkey, *args, **kwargs)
    return all_results

In [ ]:
key = jax.random.PRNGKey(42)
# Run simulations
sim_plurality = run_n_simulation(
    func=_vote_plurality,
    data=data,
    response_function=response_function_pref,
    key=key,
    n_simulations=1,
)
sim_plurality

In [ ]:
key = jax.random.PRNGKey(42)
# Run simulations
sim_plurality = run_n_simulation(
    func=_vote_quadratic,
    data=data,
    response_function=response_function_pref,
    key=key,
    n_simulations=1,
)
sim_plurality